# Evaluation Sample Selection

Selects 24 CVEs (4 per cell) from `data/rag_corpus_enriched.jsonl` for the user evaluation study.

## Cell matrix (3x2: severity x exploitability)

| | Low exploitability (EPSS < 0.5, KEV=No) | High exploitability (EPSS >= 0.5 OR KEV=Yes) |
|---|---|---|
| CRITICAL | Cell A | Cell B |
| HIGH | Cell C | Cell D |
| MEDIUM | Cell E | Cell F |

**Threshold updated 2026-07-07: EPSS >= 0.1 -> EPSS >= 0.5.** Rationale: 0.5 represents "more likely than not to be exploited" (a majority-probability threshold), a more intuitive and defensible cutoff than 0.1 for distinguishing genuinely high-likelihood exploitation from merely elevated risk. See METHODOLOGY_LOG.md Section 6 for the full analysis of this decision, including pool-size impact and the tradeoff against EPSS's right-skewed distribution.

## Selection criteria (applied within each cell)
- **Vendor diversity**: no two CVEs from the same CPE vendor within a cell, and cross-cell vendor repeats minimized across the full 24-CVE sample where possible
- **Description length spread**: mix of short (<= 250 chars) and long (> 250 chars)
- **Attack vector variety**: include non-NETWORK where possible

## Outputs
- `data/eval_sample.jsonl` — 24 selected CVEs
- `data/rag_corpus_final.jsonl` — 12,000 minus the 24 eval CVEs (use this for ChromaDB ingestion)

In [1]:
import json
from pathlib import Path
from collections import defaultdict

DATA_DIR = Path("../data")

## Load enriched corpus

In [2]:
corpus = []
with open(DATA_DIR / "rag_corpus_enriched.jsonl") as f:
    for line in f:
        corpus.append(json.loads(line))

print(f"Loaded {len(corpus):,} records")

Loaded 12,000 records


## Helper functions

In [3]:
def extract_vendor(configurations):
    """Pull vendor from the first CPE string in configurations."""
    try:
        for node in configurations[0]["nodes"]:
            for match in node.get("cpeMatch", []):
                parts = match.get("criteria", "").split(":")
                if len(parts) > 3:
                    return parts[3]
    except (IndexError, KeyError, TypeError):
        pass
    return "unknown"


def is_high_exploit(record):
    """True if EPSS >= 0.5 OR KEV-listed."""
    epss = record["epss_score"] or 0
    return epss >= 0.5 or record["kev_listed"]


def length_group(record):
    return "short" if len(record["description"]) <= 250 else "long"


def av_group(record):
    return "network" if record["attack_vector"] == "NETWORK" else "other"


def cell_label(record):
    sev = record["cvss_severity"]
    exploit = "high" if is_high_exploit(record) else "low"
    return (sev, exploit)

## Assign records to cells and show pool sizes

In [4]:
SEVERITIES = ["CRITICAL", "HIGH", "MEDIUM"]
CELL_NAMES = {
    ("CRITICAL", "low"): "A", ("CRITICAL", "high"): "B",
    ("HIGH",     "low"): "C", ("HIGH",     "high"): "D",
    ("MEDIUM",   "low"): "E", ("MEDIUM",   "high"): "F",
}

cells = defaultdict(list)
for r in corpus:
    label = cell_label(r)
    if label in CELL_NAMES:
        cells[label].append(r)

print(f"{'Cell':<6} {'Severity':<10} {'Exploitability':<16} {'Pool size':>10}")
print("-" * 46)
for sev in SEVERITIES:
    for exploit in ["low", "high"]:
        key = (sev, exploit)
        name = CELL_NAMES[key]
        print(f"{name:<6} {sev:<10} {exploit:<16} {len(cells[key]):>10,}")

Cell   Severity   Exploitability    Pool size
----------------------------------------------
A      CRITICAL   low                   1,289
B      CRITICAL   high                     66
C      HIGH       low                   4,353
D      HIGH       high                     60
E      MEDIUM     low                   5,687
F      MEDIUM     high                     33


## Selection algorithm

For each cell, picks 4 CVEs using a greedy approach:
1. Extract CPE vendor for every candidate.
2. Assign each candidate to one of four quadrants: (attack_vector_group × description_length_group).
3. Within quadrants, sort so KEV-listed appear first, then by EPSS descending.
4. Pick one CVE from each quadrant (if available), ensuring no vendor repeats.
5. Fill any remaining slots from the most-populated quadrant, maintaining vendor uniqueness.

In [5]:
def select_from_cell(candidates, n=4):
    # Attach vendor, sort each candidate so KEV-listed come first then high EPSS
    annotated = []
    for r in candidates:
        vendor = extract_vendor(r["configurations"])
        annotated.append((vendor, r))

    # Deduplicate by vendor (keep first occurrence per vendor after sorting)
    annotated.sort(key=lambda x: (not x[1]["kev_listed"], -(x[1]["epss_score"] or 0)))
    seen_vendors = set()
    vendor_deduped = []
    for vendor, r in annotated:
        if vendor not in seen_vendors:
            seen_vendors.add(vendor)
            vendor_deduped.append((vendor, r))

    # Assign to quadrants
    quadrants = {
        ("network", "short"): [],
        ("network", "long"):  [],
        ("other",   "short"): [],
        ("other",   "long"):  [],
    }
    for vendor, r in vendor_deduped:
        key = (av_group(r), length_group(r))
        quadrants[key].append((vendor, r))

    # Greedy pick: one from each quadrant first
    priority_order = [
        ("network", "short"),
        ("network", "long"),
        ("other",   "short"),
        ("other",   "long"),
    ]
    selected = []
    used_vendors = set()

    for quadrant in priority_order:
        if len(selected) >= n:
            break
        for vendor, r in quadrants[quadrant]:
            if vendor not in used_vendors:
                selected.append(r)
                used_vendors.add(vendor)
                break

    # Fill remaining from any quadrant
    for vendor, r in vendor_deduped:
        if len(selected) >= n:
            break
        if vendor not in used_vendors:
            selected.append(r)
            used_vendors.add(vendor)

    return selected

## Run selection and display results

In [6]:
selection = {}  # (sev, exploit) -> list of records

for sev in SEVERITIES:
    for exploit in ["low", "high"]:
        key = (sev, exploit)
        cell_name = CELL_NAMES[key]
        chosen = select_from_cell(cells[key], n=4)
        selection[key] = chosen

        print(f"\n=== Cell {cell_name}: {sev} / {exploit} exploitability ===")
        print(f"{'CVE ID':<20} {'KEV':>4} {'EPSS':>7} {'AV':<10} {'Len':>5} {'Vendor':<20} Description (preview)")
        print("-" * 110)
        for r in chosen:
            vendor = extract_vendor(r["configurations"])
            epss_str = f"{r['epss_score']:.4f}" if r["epss_score"] is not None else "null"
            desc_preview = r["description"][:60].replace("\n", " ") + "..."
            kev_flag = "YES" if r["kev_listed"] else "no"
            print(f"{r['id']:<20} {kev_flag:>4} {epss_str:>7} {r['attack_vector']:<10} {len(r['description']):>5} {vendor:<20} {desc_preview}")


=== Cell A: CRITICAL / low exploitability ===
CVE ID                KEV    EPSS AV           Len Vendor               Description (preview)
--------------------------------------------------------------------------------------------------------------
CVE-2026-2701          no  0.4881 NETWORK      114 progress             Authenticated user can upload a malicious file to the server...
CVE-2023-50919         no  0.4780 NETWORK      323 gl-inet              An issue was discovered on GL.iNet devices before version 4....
CVE-2023-29119         no  0.0033 ADJACENT_NETWORK   122 enelx                Waybox Enel X web management application could execute arbit...
CVE-2021-23894         no  0.0224 ADJACENT_NETWORK   290 mcafee               Deserialization of untrusted data vulnerability in McAfee Da...

=== Cell B: CRITICAL / high exploitability ===
CVE ID                KEV    EPSS AV           Len Vendor               Description (preview)
--------------------------------------------------

## Review and override

Review the table above. To swap a CVE in any cell, add the replacement ID to `OVERRIDES` below and the ID to remove to the same entry. Leave `OVERRIDES` empty to accept the algorithmic selection.

Format: `{ 'CVE-XXXX-XXXXX': 'CVE-YYYY-YYYYY' }` (old ID -> new ID)

In [7]:
# Overrides applied 2026-07-07 (EPSS threshold changed to 0.5):
#
# CVE-2026-2701 (Cell A, progress, 2026): swapped for CVE-2020-8010 (broadcom, NETWORK,
#   2020) -- removes 2026 CVE with thin RAG context, consistent with prior policy.
# CVE-2025-20362 (Cell F, cisco, 1139 chars): swapped for CVE-2022-28810 (zohocorp,
#   NETWORK) -- removes amended "Update:"-prefixed description outlier, same fix applied
#   at the 0.1 threshold.
# CVE-2024-34787 (Cell C, ivanti, LOCAL): swapped for CVE-2021-21974 (vmware,
#   ADJACENT_NETWORK) -- the tighter 0.5 threshold shrank the HIGH/low pool and
#   surfaced a new ivanti pick that duplicated Cell B and Cell D's ivanti entries.
#   Swapping also adds attack-vector diversity (ADJACENT_NETWORK) to Cell C.
# CVE-2021-28554 (Cell C, adobe, LOCAL): swapped for CVE-2021-22717 (schneider-electric,
#   NETWORK) -- same reason: removes a cross-cell repeat with Cell D's adobe entry.
# CVE-2020-2039 (Cell E, paloaltonetworks, NETWORK): swapped for CVE-2023-0157
#   (updraftplus, NETWORK) -- removes a cross-cell repeat with Cell B's paloaltonetworks
#   entry.
#
# Note: ivanti still appears in both Cell B and Cell D after these fixes. This is left
# as-is -- both are top-EPSS/KEV picks (1.000 and 0.965) representing genuinely
# near-certain exploitation, and replacing either would trade a high-signal record for
# a lower-confidence one purely for cosmetic diversity. See METHODOLOGY_LOG.md Section 6.
OVERRIDES = {
    "CVE-2026-2701": "CVE-2020-8010",
    "CVE-2025-20362": "CVE-2022-28810",
    "CVE-2024-34787": "CVE-2021-21974",
    "CVE-2021-28554": "CVE-2021-22717",
    "CVE-2020-2039": "CVE-2023-0157",
}

# Build a lookup for quick access
corpus_by_id = {r["id"]: r for r in corpus}

# Apply overrides
final_selection = {}
for key, chosen in selection.items():
    updated = []
    for r in chosen:
        if r["id"] in OVERRIDES:
            replacement_id = OVERRIDES[r["id"]]
            replacement = corpus_by_id.get(replacement_id)
            if replacement is None:
                raise ValueError(f"Override target {replacement_id!r} not found in corpus")
            print(f"Swapped {r['id']} -> {replacement_id} in cell {CELL_NAMES[key]}")
            updated.append(replacement)
        else:
            updated.append(r)
    final_selection[key] = updated

if not OVERRIDES:
    print("No overrides applied -- using algorithmic selection.")

Swapped CVE-2026-2701 -> CVE-2020-8010 in cell A
Swapped CVE-2024-34787 -> CVE-2021-21974 in cell C
Swapped CVE-2021-28554 -> CVE-2021-22717 in cell C
Swapped CVE-2020-2039 -> CVE-2023-0157 in cell E
Swapped CVE-2025-20362 -> CVE-2022-28810 in cell F


## Summary table

In [8]:
all_selected = [r for chosen in final_selection.values() for r in chosen]
eval_ids = {r["id"] for r in all_selected}

print(f"Total eval CVEs selected: {len(all_selected)}")
print(f"Unique IDs:               {len(eval_ids)}")
if len(all_selected) != len(eval_ids):
    print("WARNING: duplicate CVE IDs detected -- check overrides")

print()
print(f"{'Cell':<6} {'CVE ID':<20} {'Sev':<10} {'KEV':>4} {'EPSS':>7} {'AV':<22} {'Len':>5} {'Vendor'}")
print("-" * 90)
for sev in SEVERITIES:
    for exploit in ["low", "high"]:
        key = (sev, exploit)
        cell_name = CELL_NAMES[key]
        for r in final_selection[key]:
            vendor = extract_vendor(r["configurations"])
            epss_str = f"{r['epss_score']:.4f}" if r["epss_score"] is not None else "null"
            kev_flag = "YES" if r["kev_listed"] else "no"
            print(f"{cell_name:<6} {r['id']:<20} {r['cvss_severity']:<10} {kev_flag:>4} {epss_str:>7} {r['attack_vector']:<22} {len(r['description']):>5} {vendor}")

Total eval CVEs selected: 24
Unique IDs:               24

Cell   CVE ID               Sev         KEV    EPSS AV                       Len Vendor
------------------------------------------------------------------------------------------
A      CVE-2020-8010        CRITICAL     no  0.4867 NETWORK                  250 broadcom
A      CVE-2023-50919       CRITICAL     no  0.4780 NETWORK                  323 gl-inet
A      CVE-2023-29119       CRITICAL     no  0.0033 ADJACENT_NETWORK         122 enelx
A      CVE-2021-23894       CRITICAL     no  0.0224 ADJACENT_NETWORK         290 mcafee
B      CVE-2024-21887       CRITICAL    YES  1.0000 NETWORK                  248 ivanti
B      CVE-2021-26084       CRITICAL    YES  1.0000 NETWORK                  374 atlassian
B      CVE-2024-3400        CRITICAL    YES  1.0000 NETWORK                  399 paloaltonetworks
B      CVE-2021-42013       CRITICAL    YES  0.9996 NETWORK                  544 apache
C      CVE-2020-8958        HIGH         no

## Save eval sample

In [9]:
eval_out = DATA_DIR / "eval_sample.jsonl"
with open(eval_out, "w") as f:
    for sev in SEVERITIES:
        for exploit in ["low", "high"]:
            key = (sev, exploit)
            for r in final_selection[key]:
                record = dict(r)
                record["eval_cell"] = CELL_NAMES[key]
                record["eval_exploitability"] = exploit
                f.write(json.dumps(record) + "\n")

print(f"Saved {len(all_selected)} eval CVEs -> {eval_out}")

Saved 24 eval CVEs -> ../data/eval_sample.jsonl


## Save trimmed RAG corpus (eval CVEs removed)

This is the file to use for ChromaDB ingestion. It excludes all 24 eval CVEs to prevent data leakage during retrieval-augmented generation evaluation.

In [10]:
corpus_out = DATA_DIR / "rag_corpus_final.jsonl"
kept = [r for r in corpus if r["id"] not in eval_ids]

with open(corpus_out, "w") as f:
    for r in kept:
        f.write(json.dumps(r) + "\n")

print(f"RAG corpus (trimmed): {len(kept):,} records -> {corpus_out}")
print(f"Removed:              {len(corpus) - len(kept)} eval CVEs")
assert len(corpus) - len(kept) == len(eval_ids), "Mismatch -- some eval IDs were not in corpus"

RAG corpus (trimmed): 11,976 records -> ../data/rag_corpus_final.jsonl
Removed:              24 eval CVEs
